In [1]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
import pandas as pd
import tensorflow as tf
import keras_tuner as kt
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.tree import DecisionTreeClassifier
import json
import requests
# from config import api_key1
from pprint import pprint
import numpy as np
import psycopg2
import sqlalchemy
from sqlalchemy import create_engine, text
from sqlalchemy.orm import Session
import ipywidgets as widgets
from IPython.display import display
from sqlalchemy.ext.declarative import declarative_base
from sqlalchemy import Column, Integer, String, Float, ForeignKeyConstraint, PrimaryKeyConstraint, ForeignKey, Sequence, Date
import warnings
import time
import xgboost as xgb
from imblearn.over_sampling import SMOTE
warnings.filterwarnings('ignore')

In [2]:
url2122 ='https://www.thesportsdb.com/api/v1/json/571535/eventsseason.php?id=4328&s=2021-2022'

response2122 = requests.get(url2122)
data2122 = response2122.json()


pprint(data2122["events"])
data_df_2122 = pd.json_normalize(data2122["events"])

cols_dropped2122 = []
for col in data_df_2122.columns: 
    if ("strport" in col) | ("Thumb" in col) | ("Banner" in col) | ("Map" in col) | ("Poster" in col) | ("Square" in col) | ("Badge" in col) | ("Fanart" in col)| ("Tweet" in col) | ("Postponed" in col)| ("Group" in col) | ("Locked" in col) | ("strLeague" in col)| ("strCity" in col) | ("Local" in col) | ("strOfficial" in col) | ("strDescription" in col)| ("strTimestamp" in col) | ("intScoreVotes" in col) | ("Alternate" in col) | ("Spectators" in col)| ("strResult" in col):
        cols_dropped2122.append(col)

d_df_2122 = data_df_2122.drop(columns = cols_dropped2122, axis = 1)
d_df_2122.columns

d_df_2122

[{'dateEvent': '2021-08-13',
  'dateEventLocal': '2021-08-13',
  'idAPIfootball': '710556',
  'idAwayTeam': '133604',
  'idEvent': '1153928',
  'idHomeTeam': '134355',
  'idLeague': '4328',
  'idVenue': '18406',
  'intAwayScore': '0',
  'intHomeScore': '2',
  'intRound': '1',
  'intScore': None,
  'intScoreVotes': None,
  'intSpectators': None,
  'strAwayTeam': 'Arsenal',
  'strAwayTeamBadge': None,
  'strBanner': '',
  'strCity': '',
  'strCountry': 'England',
  'strDescriptionEN': 'Premier League newcomer Brentford plays its first-ever '
                      'game in the division against Arsenal with the Gunners '
                      'keen to start off on a positive note, after finishing '
                      'last season 8th and missing out on European soccer for '
                      'the first time since 1996.',
  'strEvent': 'Brentford vs Arsenal',
  'strEventAlternate': 'Arsenal @ Brentford',
  'strFanart': None,
  'strFilename': 'English Premier League 2021-08-13 Brentfo

,idEvent,idAPIfootball,strEvent,strFilename,strSport,idLeague,strSeason,strHomeTeam,strAwayTeam,intHomeScore,...,dateEvent,strTime,idHomeTeam,idAwayTeam,intScore,idVenue,strVenue,strCountry,strVideo,strStatus
0,1153928,710556,Brentford vs Arsenal,English Premier League 2021-08-13 Brentford vs...,Soccer,4328,2021-2022,Brentford,Arsenal,2,...,2021-08-13,19:00:00,134355,133604,None,18406,Brentford Community Stadium,England,https://www.youtube.com/watch?v=RJxHDEC-jAk,Match Finished
1,1153929,710557,Burnley vs Brighton,English Premier League 2021-08-14 Burnley vs B...,Soccer,4328,2021-2022,Burnley,Brighton,1,...,2021-08-14,14:00:00,133623,133619,None,16114,Turf Moor,England,https://www.youtube.com/watch?v=oxLS8DOcw5Y,Match Finished
2,1153930,710558,Chelsea vs Crystal Palace,English Premier League 2021-08-14 Chelsea vs C...,Soccer,4328,2021-2022,Chelsea,Crystal Palace,3,...,2021-08-14,14:00:00,133610,133632,None,16126,Stamford Bridge,England,https://www.youtube.com/watch?v=cXuUPAl-pWs,Match Finished
3,1153931,710559,Everton vs Southampton,English Premier League 2021-08-14 Everton vs S...,Soccer,4328,2021-2022,Everton,Southampton,3,...,2021-08-14,14:00:00,133615,134778,None,15532,Goodison Park,England,https://www.youtube.com/watch?v=GKulug14C2o,Match Finished
4,1153932,710560,Leicester vs Wolves,English Premier League 2021-08-14 Leicester vs...,Soccer,4328,2021-2022,Leicester,Wolves,1,...,2021-08-14,14:00:00,133626,133599,None,16116,King Power Stadium,England,https://www.youtube.com/watch?v=_Rqjh1DkSM0,Match Finished
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
375,1154303,710931,Crystal Palace vs Man United,English Premier League 2022-05-22 Crystal Pala...,Soccer,4328,2021-2022,Crystal Palace,Manchester United,1,...,2022-05-22,15:00:00,133632,133612,None,16115,Selhurst Park,England,https://www.youtube.com/watch?v=sdRq0nmPLxc,Match Finished
376,1154304,710932,Leicester vs Southampton,English Premier League 2022-05-22 Leicester vs...,Soccer,4328,2021-2022,Leicester,Southampton,4,...,2022-05-22,15:00:00,133626,134778,None,16116,King Power Stadium,England,https://www.youtube.com/watch?v=g3S3ocyVIgk,Match Finished
377,1154305,710933,Liverpool vs Wolves,English Premier League 2022-05-22 Liverpool vs...,Soccer,4328,2021-2022,Liverpool,Wolves,3,...,2022-05-22,15:00:00,133602,133599,None,15407,Anfield,England,https://www.youtube.com/watch?v=n5hqFG6taoY,Match Finished
378,1154306,710934,Man City vs Aston Villa,English Premier League 2022-05-22 Manchester C...,Soccer,4328,2021-2022,Manchester City,Aston Villa,3,...,2022-05-22,15:00:00,133613,133601,None,16119,Etihad Stadium,England,https://www.youtube.com/watch?v=cKeL28dPzIw,Match Finished


In [3]:
url2223 ='https://www.thesportsdb.com/api/v1/json/571535/eventsseason.php?id=4328&s=2022-2023'

response2223 = requests.get(url2223)
data2223 = response2223.json()


pprint(data2223["events"])
data_df_2223 = pd.json_normalize(data2223["events"])

cols_dropped2223 = []
for col in data_df_2223.columns: 
    if ("strport" in col) | ("Thumb" in col) | ("Banner" in col) | ("Map" in col) | ("Poster" in col) | ("Square" in col) | ("Badge" in col) | ("Fanart" in col)| ("Tweet" in col) | ("Postponed" in col)| ("Group" in col) | ("Locked" in col) | ("strLeague" in col)| ("strCity" in col) | ("Local" in col) | ("strOfficial" in col) | ("strDescription" in col)| ("strTimestamp" in col) | ("intScoreVotes" in col) | ("Alternate" in col) | ("Spectators" in col)| ("strResult" in col):
        cols_dropped2223.append(col)

d_df_2223 = data_df_2223.drop(columns = cols_dropped2223, axis = 1)
d_df_2223.columns

d_df_2223

[{'dateEvent': '2022-08-05',
  'dateEventLocal': '2022-08-05',
  'idAPIfootball': '867946',
  'idAwayTeam': '133604',
  'idEvent': '1570706',
  'idHomeTeam': '133632',
  'idLeague': '4328',
  'idVenue': '16115',
  'intAwayScore': '2',
  'intHomeScore': '0',
  'intRound': '1',
  'intScore': None,
  'intScoreVotes': None,
  'intSpectators': None,
  'strAwayTeam': 'Arsenal',
  'strAwayTeamBadge': None,
  'strBanner': 'https://www.thesportsdb.com/images/media/event/banner/ktydks1659896704.jpg',
  'strCity': '',
  'strCountry': 'England',
  'strDescriptionEN': 'The Premier is back! After a few eternal weeks without '
                      'club football, the best league in the world is here. '
                      'Crystal Palace and Arsenal are in charge of kicking off '
                      'this new season. A high-level match between two Premier '
                      'League classics!',
  'strEvent': 'Crystal Palace vs Arsenal',
  'strEventAlternate': 'Arsenal @ Crystal Palace',
  's

,idEvent,idAPIfootball,strEvent,strFilename,strSport,idLeague,strSeason,strHomeTeam,strAwayTeam,intHomeScore,...,dateEvent,strTime,idHomeTeam,idAwayTeam,intScore,idVenue,strVenue,strCountry,strVideo,strStatus
0,1570706,867946,Crystal Palace vs Arsenal,English Premier League 2022-08-05 Crystal Pala...,Soccer,4328,2022-2023,Crystal Palace,Arsenal,0,...,2022-08-05,19:00:00,133632,133604,None,16115,Selhurst Park,England,https://www.youtube.com/watch?v=57r-Rs5tEdY,Match Finished
1,1570707,867947,Fulham vs Liverpool,English Premier League 2022-08-06 Fulham vs Li...,Soccer,4328,2022-2023,Fulham,Liverpool,2,...,2022-08-06,11:30:00,133600,133602,None,16373,Craven Cottage,England,https://www.youtube.com/watch?v=cyReVQnoNxA,Match Finished
2,1570708,867948,Bournemouth vs Aston Villa,English Premier League 2022-08-06 Bournemouth ...,Soccer,4328,2022-2023,Bournemouth,Aston Villa,2,...,2022-08-06,14:00:00,134301,133601,None,16113,Vitality Stadium,England,https://www.youtube.com/watch?v=ltJe_PhzmXs,Match Finished
3,1570709,867949,Leeds vs Wolves,English Premier League 2022-08-06 Leeds vs Wolves,Soccer,4328,2022-2023,Leeds,Wolves,2,...,2022-08-06,14:00:00,133635,133599,None,15534,Elland Road,England,https://www.youtube.com/watch?v=h-ostlu9avE,Match Finished
4,1570712,867952,Tottenham vs Southampton,English Premier League 2022-08-06 Tottenham vs...,Soccer,4328,2022-2023,Tottenham,Southampton,4,...,2022-08-06,14:00:00,133616,134778,None,16118,Tottenham Hotspur Stadium,England,https://www.youtube.com/watch?v=laPaeEQ_Aeg,Match Finished
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
375,1571082,868322,Leeds vs Tottenham,English Premier League 2023-05-28 Leeds vs Tot...,Soccer,4328,2022-2023,Leeds,Tottenham,1,...,2023-05-28,15:30:00,133635,133616,None,15534,Elland Road,England,https://www.youtube.com/watch?v=v3buu8BOlj0,Match Finished
376,1571083,868323,Leicester vs West Ham,English Premier League 2023-05-28 Leicester vs...,Soccer,4328,2022-2023,Leicester,West Ham,2,...,2023-05-28,15:30:00,133626,133636,None,16116,King Power Stadium,England,https://www.youtube.com/watch?v=9H6Jf5Alf6c,Match Finished
377,1571084,868324,Manchester United vs Fulham,English Premier League 2023-05-28 Manchester U...,Soccer,4328,2022-2023,Manchester United,Fulham,2,...,2023-05-28,15:30:00,133612,133600,None,18332,Old Trafford,England,https://www.youtube.com/watch?v=2MR8lVFebdI,Match Finished
378,1571085,868325,Southampton vs Liverpool,English Premier League 2023-05-28 Southampton ...,Soccer,4328,2022-2023,Southampton,Liverpool,4,...,2023-05-28,15:30:00,134778,133602,None,16123,St. Mary's Stadium,England,https://www.youtube.com/watch?v=AAseWAvsHAw,Match Finished


In [4]:
url2324 ='https://www.thesportsdb.com/api/v1/json/571535/eventsseason.php?id=4328&s=2023-2024'

response2324 = requests.get(url2324)
data2324 = response2324.json()


pprint(data2324["events"])
data_df2324 = pd.json_normalize(data2324["events"])

cols_dropped2324 = []
for col in data_df2324.columns: 
    if ("strport" in col) | ("Thumb" in col) | ("Banner" in col) | ("Map" in col) | ("Poster" in col) | ("Square" in col) | ("Badge" in col) | ("Fanart" in col)| ("Tweet" in col) | ("Postponed" in col)| ("Group" in col) | ("Locked" in col) | ("strLeague" in col)| ("strCity" in col) | ("Local" in col) | ("strOfficial" in col) | ("strDescription" in col)| ("strTimestamp" in col) | ("intScoreVotes" in col) | ("Alternate" in col) | ("Spectators" in col)| ("strResult" in col):
        cols_dropped2324.append(col)

d_df_2324 = data_df2324.drop(columns = cols_dropped2324, axis = 1)
d_df_2324.columns

d_df_2324

[{'dateEvent': '2023-08-11',
  'dateEventLocal': '2023-08-11',
  'idAPIfootball': '1035037',
  'idAwayTeam': '133613',
  'idEvent': '1818395',
  'idHomeTeam': '133623',
  'idLeague': '4328',
  'idVenue': '16114',
  'intAwayScore': '3',
  'intHomeScore': '0',
  'intRound': '1',
  'intScore': None,
  'intScoreVotes': None,
  'intSpectators': None,
  'strAwayTeam': 'Manchester City',
  'strAwayTeamBadge': 'https://www.thesportsdb.com/images/media/team/badge/vwpvry1467462651.png',
  'strBanner': 'https://www.thesportsdb.com/images/media/event/banner/wsg2rj1689062918.jpg',
  'strCity': '',
  'strCountry': 'England',
  'strDescriptionEN': '',
  'strEvent': 'Burnley vs Manchester City',
  'strEventAlternate': 'Manchester City @ Burnley',
  'strFanart': None,
  'strFilename': 'English Premier League 2023-08-11 Burnley vs Manchester City',
  'strGroup': None,
  'strHomeTeam': 'Burnley',
  'strHomeTeamBadge': 'https://www.thesportsdb.com/images/media/team/badge/ql7nl31686893820.png',
  'strLeagu

,idEvent,idAPIfootball,strEvent,strFilename,strSport,idLeague,strSeason,strHomeTeam,strAwayTeam,intHomeScore,...,dateEvent,strTime,idHomeTeam,idAwayTeam,intScore,idVenue,strVenue,strCountry,strVideo,strStatus
0,1818395,1035037,Burnley vs Manchester City,English Premier League 2023-08-11 Burnley vs M...,Soccer,4328,2023-2024,Burnley,Manchester City,0,...,2023-08-11,19:00:00,133623,133613,None,16114,Turf Moor,England,https://www.youtube.com/watch?v=HFlHTG8wUKQ,Match Finished
1,1818396,1035038,Arsenal vs Nottingham Forest,English Premier League 2023-08-12 Arsenal vs N...,Soccer,4328,2023-2024,Arsenal,Nottingham Forest,2,...,2023-08-12,11:30:00,133604,133720,None,15528,Emirates Stadium,England,https://www.youtube.com/watch?v=xMrC3TE3rtE,Match Finished
2,1818397,1035039,Bournemouth vs West Ham,English Premier League 2023-08-12 Bournemouth ...,Soccer,4328,2023-2024,Bournemouth,West Ham,1,...,2023-08-12,14:00:00,134301,133636,None,16113,Vitality Stadium,England,https://www.youtube.com/watch?v=c_D074QE_to,Match Finished
3,1818398,1035040,Brighton vs Luton,English Premier League 2023-08-12 Brighton vs ...,Soccer,4328,2023-2024,Brighton,Luton,4,...,2023-08-12,14:00:00,133619,133888,None,16124,The American Express Community Stadium,England,https://www.youtube.com/watch?v=NvHbtF8hUC4,Match Finished
4,1818399,1035041,Everton vs Fulham,English Premier League 2023-08-12 Everton vs F...,Soccer,4328,2023-2024,Everton,Fulham,0,...,2023-08-12,14:00:00,133615,133600,None,15532,Goodison Park,England,https://www.youtube.com/watch?v=cvMAhxkPE48,Match Finished
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
375,1818776,1035549,Crystal Palace vs Aston Villa,English Premier League 2024-05-19 Crystal Pala...,Soccer,4328,2023-2024,Crystal Palace,Aston Villa,5,...,2024-05-19,15:00:00,133632,133601,None,16115,Selhurst Park,England,https://www.youtube.com/watch?v=JEY0GBRS39o,Match Finished
376,1818777,1035550,Liverpool vs Wolves,English Premier League 2024-05-19 Liverpool vs...,Soccer,4328,2023-2024,Liverpool,Wolves,2,...,2024-05-19,15:00:00,133602,133599,None,15407,Anfield,England,https://www.youtube.com/watch?v=C5pv1o8ZSok,Match Finished
377,1818778,1035551,Luton vs Fulham,English Premier League 2024-05-19 Luton vs Fulham,Soccer,4328,2023-2024,Luton,Fulham,2,...,2024-05-19,15:00:00,133888,133600,None,18407,Kenilworth Road,England,https://www.youtube.com/watch?v=G0MfkSUMNCk,Match Finished
378,1818779,1035552,Manchester City vs West Ham,English Premier League 2024-05-19 Manchester C...,Soccer,4328,2023-2024,Manchester City,West Ham,3,...,2024-05-19,15:00:00,133613,133636,None,16119,Etihad Stadium,England,https://www.youtube.com/watch?v=9oJu2OBV62w,Match Finished


In [5]:
url2425 ='https://www.thesportsdb.com/api/v1/json/571535/eventsseason.php?id=4328&s=2024-2025'

response2425 = requests.get(url2425)
data2425 = response2425.json()


pprint(data2425["events"])
data_df_2425 = pd.json_normalize(data2425["events"])

cols_dropped_2425 = []
for col in data_df_2425.columns: 
    if ("strport" in col) | ("Thumb" in col) | ("Banner" in col) | ("Map" in col) | ("Poster" in col) | ("Square" in col) | ("Badge" in col) | ("Fanart" in col)| ("Tweet" in col) | ("Postponed" in col)| ("Group" in col) | ("Locked" in col) | ("strLeague" in col)| ("strCity" in col) | ("Local" in col) | ("strOfficial" in col) | ("strDescription" in col)| ("strTimestamp" in col) | ("intScoreVotes" in col) | ("Alternate" in col) | ("Spectators" in col)| ("strResult" in col):
        cols_dropped_2425.append(col)

d_df_2425 = data_df_2425.drop(columns = cols_dropped_2425, axis = 1)
d_df_2425.columns

d_df_2425

[{'dateEvent': '2024-08-16',
  'dateEventLocal': '2024-08-16',
  'idAPIfootball': '1208021',
  'idAwayTeam': '133600',
  'idEvent': '2069556',
  'idHomeTeam': '133612',
  'idLeague': '4328',
  'idVenue': '18332',
  'intAwayScore': '0',
  'intHomeScore': '1',
  'intRound': '1',
  'intScore': None,
  'intScoreVotes': None,
  'intSpectators': None,
  'strAwayTeam': 'Fulham',
  'strAwayTeamBadge': 'https://www.thesportsdb.com/images/media/team/badge/xwwvyt1448811086.png',
  'strBanner': 'https://www.thesportsdb.com/images/media/event/banner/rzkftq1718956414.jpg',
  'strCity': '',
  'strCountry': 'England',
  'strDescriptionEN': '',
  'strEvent': 'Manchester United vs Fulham',
  'strEventAlternate': 'Fulham @ Manchester United',
  'strFanart': None,
  'strFilename': 'English Premier League 2024-08-16 Manchester United vs '
                 'Fulham',
  'strGroup': '',
  'strHomeTeam': 'Manchester United',
  'strHomeTeamBadge': 'https://www.thesportsdb.com/images/media/team/badge/xzqdr1151766

,idEvent,idAPIfootball,strEvent,strFilename,strSport,idLeague,strSeason,strHomeTeam,strAwayTeam,intHomeScore,...,dateEvent,strTime,idHomeTeam,idAwayTeam,intScore,idVenue,strVenue,strCountry,strVideo,strStatus
0,2069556,1208021,Manchester United vs Fulham,English Premier League 2024-08-16 Manchester U...,Soccer,4328,2024-2025,Manchester United,Fulham,1,...,2024-08-16,19:00:00,133612,133600,None,18332,Old Trafford,England,https://www.youtube.com/watch?v=5VM9zKyfk9E,Match Finished
1,2069557,1208022,Ipswich vs Liverpool,English Premier League 2024-08-17 Ipswich vs L...,Soccer,4328,2024-2025,Ipswich,Liverpool,0,...,2024-08-17,11:30:00,133622,133602,None,16258,Portman Road,England,https://www.youtube.com/watch?v=QvJz32xg5_w,Match Finished
2,2069558,1208023,Arsenal vs Wolves,English Premier League 2024-08-17 Arsenal vs W...,Soccer,4328,2024-2025,Arsenal,Wolves,2,...,2024-08-17,14:00:00,133604,133599,None,15528,Emirates Stadium,England,https://www.youtube.com/watch?v=bPUDxEV5em8,Match Finished
3,2069559,1208024,Everton vs Brighton,English Premier League 2024-08-17 Everton vs B...,Soccer,4328,2024-2025,Everton,Brighton,0,...,2024-08-17,14:00:00,133615,133619,None,15532,Goodison Park,England,https://www.youtube.com/watch?v=1E2BUiNs_u0,Match Finished
4,2069560,1208025,Newcastle vs Southampton,English Premier League 2024-08-17 Newcastle vs...,Soccer,4328,2024-2025,Newcastle,Southampton,1,...,2024-08-17,14:00:00,134777,134778,None,16294,St James Park,England,https://www.youtube.com/watch?v=0_boaYEPy5Q,Match Finished
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
375,2070213,1208398,Newcastle vs Everton,English Premier League 2025-05-25 Newcastle vs...,Soccer,4328,2024-2025,Newcastle,Everton,None,...,2025-05-25,15:00:00,134777,133615,None,16294,St James Park,England,None,Not Started
376,2070214,1208399,Nottingham Forest vs Chelsea,English Premier League 2025-05-25 Nottingham F...,Soccer,4328,2024-2025,Nottingham Forest,Chelsea,None,...,2025-05-25,15:00:00,133720,133610,None,18249,City Ground,England,None,Not Started
377,2070215,1208400,Southampton vs Arsenal,English Premier League 2025-05-25 Southampton ...,Soccer,4328,2024-2025,Southampton,Arsenal,None,...,2025-05-25,15:00:00,134778,133604,None,16123,St. Mary,England,None,Not Started
378,2070216,1208401,Tottenham vs Brighton,English Premier League 2025-05-25 Tottenham vs...,Soccer,4328,2024-2025,Tottenham,Brighton,None,...,2025-05-25,15:00:00,133616,133619,None,16118,Tottenham Hotspur Stadium,England,None,Not Started


In [6]:
d_df = pd.concat([d_df_2223, d_df_2324,d_df_2425])
                 
d_df

,idEvent,idAPIfootball,strEvent,strFilename,strSport,idLeague,strSeason,strHomeTeam,strAwayTeam,intHomeScore,...,dateEvent,strTime,idHomeTeam,idAwayTeam,intScore,idVenue,strVenue,strCountry,strVideo,strStatus
0,1570706,867946,Crystal Palace vs Arsenal,English Premier League 2022-08-05 Crystal Pala...,Soccer,4328,2022-2023,Crystal Palace,Arsenal,0,...,2022-08-05,19:00:00,133632,133604,None,16115,Selhurst Park,England,https://www.youtube.com/watch?v=57r-Rs5tEdY,Match Finished
1,1570707,867947,Fulham vs Liverpool,English Premier League 2022-08-06 Fulham vs Li...,Soccer,4328,2022-2023,Fulham,Liverpool,2,...,2022-08-06,11:30:00,133600,133602,None,16373,Craven Cottage,England,https://www.youtube.com/watch?v=cyReVQnoNxA,Match Finished
2,1570708,867948,Bournemouth vs Aston Villa,English Premier League 2022-08-06 Bournemouth ...,Soccer,4328,2022-2023,Bournemouth,Aston Villa,2,...,2022-08-06,14:00:00,134301,133601,None,16113,Vitality Stadium,England,https://www.youtube.com/watch?v=ltJe_PhzmXs,Match Finished
3,1570709,867949,Leeds vs Wolves,English Premier League 2022-08-06 Leeds vs Wolves,Soccer,4328,2022-2023,Leeds,Wolves,2,...,2022-08-06,14:00:00,133635,133599,None,15534,Elland Road,England,https://www.youtube.com/watch?v=h-ostlu9avE,Match Finished
4,1570712,867952,Tottenham vs Southampton,English Premier League 2022-08-06 Tottenham vs...,Soccer,4328,2022-2023,Tottenham,Southampton,4,...,2022-08-06,14:00:00,133616,134778,None,16118,Tottenham Hotspur Stadium,England,https://www.youtube.com/watch?v=laPaeEQ_Aeg,Match Finished
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
375,2070213,1208398,Newcastle vs Everton,English Premier League 2025-05-25 Newcastle vs...,Soccer,4328,2024-2025,Newcastle,Everton,None,...,2025-05-25,15:00:00,134777,133615,None,16294,St James Park,England,None,Not Started
376,2070214,1208399,Nottingham Forest vs Chelsea,English Premier League 2025-05-25 Nottingham F...,Soccer,4328,2024-2025,Nottingham Forest,Chelsea,None,...,2025-05-25,15:00:00,133720,133610,None,18249,City Ground,England,None,Not Started
377,2070215,1208400,Southampton vs Arsenal,English Premier League 2025-05-25 Southampton ...,Soccer,4328,2024-2025,Southampton,Arsenal,None,...,2025-05-25,15:00:00,134778,133604,None,16123,St. Mary,England,None,Not Started
378,2070216,1208401,Tottenham vs Brighton,English Premier League 2025-05-25 Tottenham vs...,Soccer,4328,2024-2025,Tottenham,Brighton,None,...,2025-05-25,15:00:00,133616,133619,None,16118,Tottenham Hotspur Stadium,England,None,Not Started


In [7]:
d_df2 = d_df.loc[d_df["strStatus"] == "Match Finished"].reset_index(drop = True)

d_df2

,idEvent,idAPIfootball,strEvent,strFilename,strSport,idLeague,strSeason,strHomeTeam,strAwayTeam,intHomeScore,...,dateEvent,strTime,idHomeTeam,idAwayTeam,intScore,idVenue,strVenue,strCountry,strVideo,strStatus
0,1570706,867946,Crystal Palace vs Arsenal,English Premier League 2022-08-05 Crystal Pala...,Soccer,4328,2022-2023,Crystal Palace,Arsenal,0,...,2022-08-05,19:00:00,133632,133604,None,16115,Selhurst Park,England,https://www.youtube.com/watch?v=57r-Rs5tEdY,Match Finished
1,1570707,867947,Fulham vs Liverpool,English Premier League 2022-08-06 Fulham vs Li...,Soccer,4328,2022-2023,Fulham,Liverpool,2,...,2022-08-06,11:30:00,133600,133602,None,16373,Craven Cottage,England,https://www.youtube.com/watch?v=cyReVQnoNxA,Match Finished
2,1570708,867948,Bournemouth vs Aston Villa,English Premier League 2022-08-06 Bournemouth ...,Soccer,4328,2022-2023,Bournemouth,Aston Villa,2,...,2022-08-06,14:00:00,134301,133601,None,16113,Vitality Stadium,England,https://www.youtube.com/watch?v=ltJe_PhzmXs,Match Finished
3,1570709,867949,Leeds vs Wolves,English Premier League 2022-08-06 Leeds vs Wolves,Soccer,4328,2022-2023,Leeds,Wolves,2,...,2022-08-06,14:00:00,133635,133599,None,15534,Elland Road,England,https://www.youtube.com/watch?v=h-ostlu9avE,Match Finished
4,1570712,867952,Tottenham vs Southampton,English Premier League 2022-08-06 Tottenham vs...,Soccer,4328,2022-2023,Tottenham,Southampton,4,...,2022-08-06,14:00:00,133616,134778,None,16118,Tottenham Hotspur Stadium,England,https://www.youtube.com/watch?v=laPaeEQ_Aeg,Match Finished
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
974,2070051,1208236,Everton vs Tottenham,English Premier League 2025-01-19 Everton vs T...,Soccer,4328,2024-2025,Everton,Tottenham,3,...,2025-01-19,14:00:00,133615,133616,None,15532,Goodison Park,England,https://www.youtube.com/watch?v=jBt6xuZ-siE,Match Finished
975,2070052,1208237,Ipswich vs Manchester City,English Premier League 2025-01-19 Ipswich vs M...,Soccer,4328,2024-2025,Ipswich,Manchester City,0,...,2025-01-19,16:30:00,133622,133613,None,16258,Portman Road,England,https://www.youtube.com/watch?v=1Mx2VhJaN1U,Match Finished
976,2070054,1208239,Manchester United vs Brighton,English Premier League 2025-01-19 Manchester U...,Soccer,4328,2024-2025,Manchester United,Brighton,1,...,2025-01-19,14:00:00,133612,133619,None,18332,Old Trafford,England,https://www.youtube.com/watch?v=Ej7-0uHGC18,Match Finished
977,2070056,1208241,Nottingham Forest vs Southampton,English Premier League 2025-01-19 Nottingham F...,Soccer,4328,2024-2025,Nottingham Forest,Southampton,3,...,2025-01-19,14:00:00,133720,134778,None,18249,City Ground,England,https://www.youtube.com/watch?v=9NJr7jgVY8A,Match Finished


In [8]:
# df_part_2223 = d_df_2223.iloc[99:199]

event_ids = list(d_df2["idEvent"])  # Example event IDs

eventstats = []

# Loop over each event and fetch statistics
for event_id in event_ids:
    url = f"https://www.thesportsdb.com/api/v1/json/571535/lookupeventstats.php?id={event_id}"
    
    # Make the API call
    response = requests.get(url)
    
    # Parse the JSON response
    data = response.json()

    pprint(data)
    
    if data['eventstats'] is None:
        print(f"No event stats available for event ID {event_id}")
        continue  # Skip this event or handle accordingly
    else:
        # Proceed with processing if event stats are available
        eventstats.extend(data['eventstats'])

    time.sleep(0.6)
    

matches = {}

# Step 2: Iterate through the eventstats and structure the data
for stat in eventstats:
    match_id = stat['idEvent']  # Get the match name
    
    # If this match is not already in the dictionary, create a new entry for it
    if match_id not in matches:
        matches[match_id] = {'match': match_id}
    
    # Add the statistic to the dictionary for this match
    stat_name = stat['strStat']
    stat_home = stat['intHome']
    stat_away = stat['intAway']
    
    # Dynamically add the statistics for Home and Away
    matches[match_id][f'{stat_name} Home'] = stat_home
    matches[match_id][f'{stat_name} Away'] = stat_away

# Step 3: Convert the dictionary into a DataFrame
event_df_2223_2 = pd.DataFrame.from_dict(matches, orient='index').reset_index()
event_df_2223_2 = event_df_2223_2.drop(columns = ["index"])

# Step 4: Show the DataFrame
event_df_2223_2.rename(columns = {"match": "idEvent"}, inplace = True)
event_df_2223_2

{'eventstats': [{'idApiFootball': '867946',
                 'idEvent': '1570706',
                 'idStatistic': '99852',
                 'intAway': '2',
                 'intHome': '2',
                 'strEvent': 'Crystal Palace vs Arsenal',
                 'strStat': 'Shots on Goal'},
                {'idApiFootball': '867946',
                 'idEvent': '1570706',
                 'idStatistic': '99853',
                 'intAway': '4',
                 'intHome': '2',
                 'strEvent': 'Crystal Palace vs Arsenal',
                 'strStat': 'Shots off Goal'},
                {'idApiFootball': '867946',
                 'idEvent': '1570706',
                 'idStatistic': '99854',
                 'intAway': '10',
                 'intHome': '10',
                 'strEvent': 'Crystal Palace vs Arsenal',
                 'strStat': 'Total Shots'},
                {'idApiFootball': '867946',
                 'idEvent': '1570706',
                 'idStatistic': '9

,idEvent,Shots on Goal Home,Shots on Goal Away,Shots off Goal Home,Shots off Goal Away,Total Shots Home,Total Shots Away,Blocked Shots Home,Blocked Shots Away,Shots insidebox Home,...,Total passes Home,Total passes Away,Passes accurate Home,Passes accurate Away,Passes % Home,Passes % Away,expected_goals Home,expected_goals Away,goals_prevented Home,goals_prevented Away
0,1570706,2,2,2,4,10,10,6,4,9,...,532,427,460,349,86,82,NaN,NaN,NaN,NaN
1,1570707,3,4,2,5,9,11,4,2,7,...,288,606,178,476,62,79,NaN,NaN,NaN,NaN
2,1570708,3,2,2,7,7,15,2,6,6,...,298,570,209,481,70,84,NaN,NaN,NaN,NaN
3,1570709,4,6,5,6,12,15,3,3,7,...,340,537,255,455,75,85,NaN,NaN,NaN,NaN
4,1570712,8,2,6,5,18,9,4,2,14,...,546,407,465,332,85,82,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
962,2070057,0,7,4,4,7,11,3,0,4,...,535,465,441,378,82,81,0,1,0,0
963,2070051,6,6,4,2,12,11,2,3,9,...,328,600,269,518,82,86,2,1,-1,-1
964,2070052,4,9,1,3,8,17,3,5,5,...,347,742,302,682,87,92,1,3,1,1
965,2070054,1,3,3,1,10,6,6,2,9,...,451,433,370,353,82,82,1,2,0,0


In [9]:
d_df2.reset_index(drop = True)

,idEvent,idAPIfootball,strEvent,strFilename,strSport,idLeague,strSeason,strHomeTeam,strAwayTeam,intHomeScore,...,dateEvent,strTime,idHomeTeam,idAwayTeam,intScore,idVenue,strVenue,strCountry,strVideo,strStatus
0,1570706,867946,Crystal Palace vs Arsenal,English Premier League 2022-08-05 Crystal Pala...,Soccer,4328,2022-2023,Crystal Palace,Arsenal,0,...,2022-08-05,19:00:00,133632,133604,None,16115,Selhurst Park,England,https://www.youtube.com/watch?v=57r-Rs5tEdY,Match Finished
1,1570707,867947,Fulham vs Liverpool,English Premier League 2022-08-06 Fulham vs Li...,Soccer,4328,2022-2023,Fulham,Liverpool,2,...,2022-08-06,11:30:00,133600,133602,None,16373,Craven Cottage,England,https://www.youtube.com/watch?v=cyReVQnoNxA,Match Finished
2,1570708,867948,Bournemouth vs Aston Villa,English Premier League 2022-08-06 Bournemouth ...,Soccer,4328,2022-2023,Bournemouth,Aston Villa,2,...,2022-08-06,14:00:00,134301,133601,None,16113,Vitality Stadium,England,https://www.youtube.com/watch?v=ltJe_PhzmXs,Match Finished
3,1570709,867949,Leeds vs Wolves,English Premier League 2022-08-06 Leeds vs Wolves,Soccer,4328,2022-2023,Leeds,Wolves,2,...,2022-08-06,14:00:00,133635,133599,None,15534,Elland Road,England,https://www.youtube.com/watch?v=h-ostlu9avE,Match Finished
4,1570712,867952,Tottenham vs Southampton,English Premier League 2022-08-06 Tottenham vs...,Soccer,4328,2022-2023,Tottenham,Southampton,4,...,2022-08-06,14:00:00,133616,134778,None,16118,Tottenham Hotspur Stadium,England,https://www.youtube.com/watch?v=laPaeEQ_Aeg,Match Finished
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
974,2070051,1208236,Everton vs Tottenham,English Premier League 2025-01-19 Everton vs T...,Soccer,4328,2024-2025,Everton,Tottenham,3,...,2025-01-19,14:00:00,133615,133616,None,15532,Goodison Park,England,https://www.youtube.com/watch?v=jBt6xuZ-siE,Match Finished
975,2070052,1208237,Ipswich vs Manchester City,English Premier League 2025-01-19 Ipswich vs M...,Soccer,4328,2024-2025,Ipswich,Manchester City,0,...,2025-01-19,16:30:00,133622,133613,None,16258,Portman Road,England,https://www.youtube.com/watch?v=1Mx2VhJaN1U,Match Finished
976,2070054,1208239,Manchester United vs Brighton,English Premier League 2025-01-19 Manchester U...,Soccer,4328,2024-2025,Manchester United,Brighton,1,...,2025-01-19,14:00:00,133612,133619,None,18332,Old Trafford,England,https://www.youtube.com/watch?v=Ej7-0uHGC18,Match Finished
977,2070056,1208241,Nottingham Forest vs Southampton,English Premier League 2025-01-19 Nottingham F...,Soccer,4328,2024-2025,Nottingham Forest,Southampton,3,...,2025-01-19,14:00:00,133720,134778,None,18249,City Ground,England,https://www.youtube.com/watch?v=9NJr7jgVY8A,Match Finished


In [10]:
df_full_merge = pd.merge(d_df2, event_df_2223_2, on = "idEvent")

df_full_merge

,idEvent,idAPIfootball,strEvent,strFilename,strSport,idLeague,strSeason,strHomeTeam,strAwayTeam,intHomeScore,...,Total passes Home,Total passes Away,Passes accurate Home,Passes accurate Away,Passes % Home,Passes % Away,expected_goals Home,expected_goals Away,goals_prevented Home,goals_prevented Away
0,1570706,867946,Crystal Palace vs Arsenal,English Premier League 2022-08-05 Crystal Pala...,Soccer,4328,2022-2023,Crystal Palace,Arsenal,0,...,532,427,460,349,86,82,NaN,NaN,NaN,NaN
1,1570707,867947,Fulham vs Liverpool,English Premier League 2022-08-06 Fulham vs Li...,Soccer,4328,2022-2023,Fulham,Liverpool,2,...,288,606,178,476,62,79,NaN,NaN,NaN,NaN
2,1570708,867948,Bournemouth vs Aston Villa,English Premier League 2022-08-06 Bournemouth ...,Soccer,4328,2022-2023,Bournemouth,Aston Villa,2,...,298,570,209,481,70,84,NaN,NaN,NaN,NaN
3,1570709,867949,Leeds vs Wolves,English Premier League 2022-08-06 Leeds vs Wolves,Soccer,4328,2022-2023,Leeds,Wolves,2,...,340,537,255,455,75,85,NaN,NaN,NaN,NaN
4,1570712,867952,Tottenham vs Southampton,English Premier League 2022-08-06 Tottenham vs...,Soccer,4328,2022-2023,Tottenham,Southampton,4,...,546,407,465,332,85,82,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
962,2070057,1208242,West Ham vs Crystal Palace,English Premier League 2025-01-18 West Ham vs ...,Soccer,4328,2024-2025,West Ham,Crystal Palace,0,...,535,465,441,378,82,81,0,1,0,0
963,2070051,1208236,Everton vs Tottenham,English Premier League 2025-01-19 Everton vs T...,Soccer,4328,2024-2025,Everton,Tottenham,3,...,328,600,269,518,82,86,2,1,-1,-1
964,2070052,1208237,Ipswich vs Manchester City,English Premier League 2025-01-19 Ipswich vs M...,Soccer,4328,2024-2025,Ipswich,Manchester City,0,...,347,742,302,682,87,92,1,3,1,1
965,2070054,1208239,Manchester United vs Brighton,English Premier League 2025-01-19 Manchester U...,Soccer,4328,2024-2025,Manchester United,Brighton,1,...,451,433,370,353,82,82,1,2,0,0


In [11]:
cols_dropped = []
for col in df_full_merge.columns: 
    if ("strport" in col) | ("Thumb" in col) | ("Banner" in col) | ("Map" in col) | ("Poster" in col) | ("Square" in col) | ("Badge" in col) | ("Fanart" in col)| ("Tweet" in col) | ("Postponed" in col)| ("Group" in col) | ("Locked" in col) | ("strLeague" in col)| ("strCity" in col) | ("Local" in col) | ("strOfficial" in col) | ("strDescription" in col)| ("strTimestamp" in col) | ("intScoreVotes" in col) | ("Alternate" in col) | ("Spectators" in col)| ("strResult" in col) | ("id" in col)| ("strS" in col) | ("intS" in col)| ("Filename" in col) | ("strCountry" in col):
        cols_dropped.append(col)

d_df_full2 = df_full_merge.drop(columns = cols_dropped, axis = 1)

d_df_full2 = d_df_full2.drop(columns = ["goals_prevented Home", "goals_prevented Away"])
d_df_full2 = d_df_full2.dropna()

len(d_df_full2)

771

In [12]:
for index, row in d_df_full2.iterrows():
    if row["intHomeScore"] > row["intAwayScore"]:
        d_df_full2.at[index, "winner"] = row["strHomeTeam"]
    elif row["intHomeScore"] < row["intAwayScore"]:
        d_df_full2.at[index, "winner"] = row["strAwayTeam"]
    else:
        d_df_full2.at[index, "winner"] = "draw"

d_df_full2

,strEvent,strHomeTeam,strAwayTeam,intHomeScore,intRound,intAwayScore,dateEvent,strTime,strVenue,Shots on Goal Home,...,Goalkeeper Saves Away,Total passes Home,Total passes Away,Passes accurate Home,Passes accurate Away,Passes % Home,Passes % Away,expected_goals Home,expected_goals Away,winner
185,Leicester vs Brighton,Leicester,Brighton,2,21,2,2023-01-21,15:00:00,King Power Stadium,2,...,0,359,636,287,562,80,88,1,2,draw
187,Southampton vs Aston Villa,Southampton,Aston Villa,0,21,1,2023-01-21,15:00:00,St. Mary's Stadium,5,...,5,319,556,223,447,70,80,0,1,Aston Villa
188,West Ham vs Everton,West Ham,Everton,2,21,0,2023-01-21,15:00:00,London Stadium,4,...,2,341,713,271,623,79,87,2,1,West Ham
190,Arsenal vs Manchester United,Arsenal,Manchester United,3,21,2,2023-01-22,16:30:00,Emirates Stadium,5,...,2,517,386,425,300,82,78,3,0,Arsenal
191,Leeds vs Brentford,Leeds,Brentford,0,21,0,2023-01-22,14:00:00,Elland Road,6,...,6,496,341,366,220,74,65,1,0,draw
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
962,West Ham vs Crystal Palace,West Ham,Crystal Palace,0,22,2,2025-01-18,15:00:00,London Stadium,0,...,0,535,465,441,378,82,81,0,1,Crystal Palace
963,Everton vs Tottenham,Everton,Tottenham,3,22,2,2025-01-19,14:00:00,Goodison Park,6,...,4,328,600,269,518,82,86,2,1,Everton
964,Ipswich vs Manchester City,Ipswich,Manchester City,0,22,6,2025-01-19,16:30:00,Portman Road,4,...,4,347,742,302,682,87,92,1,3,Manchester City
965,Manchester United vs Brighton,Manchester United,Brighton,1,22,3,2025-01-19,14:00:00,Old Trafford,1,...,0,451,433,370,353,82,82,1,2,Brighton


In [13]:
for index, row in d_df_full2.iterrows(): 
    if row["strHomeTeam"] == row["winner"]:
        d_df_full2.at[index, "homeResult"] = 1
    elif (row["strHomeTeam"] != row["winner"] and (row["winner"] != "draw")):
        d_df_full2.at[index, "homeResult"] = -1
    elif row["winner"] == "draw": 
        d_df_full2.at[index, "homeResult"] = 0
        
# d_df_full2 = d_df_full2.drop(columns = "result", axis = 1)

d_df_full2
        

,strEvent,strHomeTeam,strAwayTeam,intHomeScore,intRound,intAwayScore,dateEvent,strTime,strVenue,Shots on Goal Home,...,Total passes Home,Total passes Away,Passes accurate Home,Passes accurate Away,Passes % Home,Passes % Away,expected_goals Home,expected_goals Away,winner,homeResult
185,Leicester vs Brighton,Leicester,Brighton,2,21,2,2023-01-21,15:00:00,King Power Stadium,2,...,359,636,287,562,80,88,1,2,draw,0.0
187,Southampton vs Aston Villa,Southampton,Aston Villa,0,21,1,2023-01-21,15:00:00,St. Mary's Stadium,5,...,319,556,223,447,70,80,0,1,Aston Villa,-1.0
188,West Ham vs Everton,West Ham,Everton,2,21,0,2023-01-21,15:00:00,London Stadium,4,...,341,713,271,623,79,87,2,1,West Ham,1.0
190,Arsenal vs Manchester United,Arsenal,Manchester United,3,21,2,2023-01-22,16:30:00,Emirates Stadium,5,...,517,386,425,300,82,78,3,0,Arsenal,1.0
191,Leeds vs Brentford,Leeds,Brentford,0,21,0,2023-01-22,14:00:00,Elland Road,6,...,496,341,366,220,74,65,1,0,draw,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
962,West Ham vs Crystal Palace,West Ham,Crystal Palace,0,22,2,2025-01-18,15:00:00,London Stadium,0,...,535,465,441,378,82,81,0,1,Crystal Palace,-1.0
963,Everton vs Tottenham,Everton,Tottenham,3,22,2,2025-01-19,14:00:00,Goodison Park,6,...,328,600,269,518,82,86,2,1,Everton,1.0
964,Ipswich vs Manchester City,Ipswich,Manchester City,0,22,6,2025-01-19,16:30:00,Portman Road,4,...,347,742,302,682,87,92,1,3,Manchester City,-1.0
965,Manchester United vs Brighton,Manchester United,Brighton,1,22,3,2025-01-19,14:00:00,Old Trafford,1,...,451,433,370,353,82,82,1,2,Brighton,-1.0


In [14]:
# d_df_full2['dateEvent'] = pd.to_datetime(d_df_full2['dateEvent'])
# d_df_full2 = d_df_full2.sort_values('dateEvent')

# # Initialize new columns for win streaks (last 5 games)
# d_df_full2["win streak home"] = 0
# d_df_full2["win streak away"] = 0

# # Create a dictionary to track win streaks for each team
# team_streaks = {}

# # Iterate over the dataframe
# for i, row in d_df_full2.iterrows():
#     home_team = row['strHomeTeam']
#     away_team = row['strAwayTeam']
#     winner = row['winner']
    
#     # For home team
#     if home_team not in team_streaks:
#         team_streaks[home_team] = {'streaks': [], 'current_streak': 0}
    
#     # Track home team's win/loss in the last 5 games
#     if winner == home_team:
#         team_streaks[home_team]['streaks'].append(1)  # Win
#     elif winner is None:  # If draw
#         team_streaks[home_team]['streaks'].append(0)  # Draw
#     else:
#         team_streaks[home_team]['streaks'].append(-1)  # Loss
    
#     # Keep the last 5 matches for the home team
#     if len(team_streaks[home_team]['streaks']) > 5:
#         team_streaks[home_team]['streaks'].pop(0)
    
#     # Update the win streak column for the home team
#     d_df_full2.at[i, "win streak home"] = sum(1 for x in team_streaks[home_team]['streaks'] if x == 1)
    
#     # For away team
#     if away_team not in team_streaks:
#         team_streaks[away_team] = {'streaks': [], 'current_streak': 0}
    
#     # Track away team's win/loss in the last 5 games
#     if winner == away_team:
#         team_streaks[away_team]['streaks'].append(1)  # Win
#     elif winner is None:  # If draw
#         team_streaks[away_team]['streaks'].append(0)  # Draw
#     else:
#         team_streaks[away_team]['streaks'].append(-1)  # Loss
    
#     # Keep the last 5 matches for the away team
#     if len(team_streaks[away_team]['streaks']) > 5:
#         team_streaks[away_team]['streaks'].pop(0)
    
#     # Update the win streak column for the away team
#     d_df_full2.at[i, "win streak away"] = sum(1 for x in team_streaks[away_team]['streaks'] if x == 1)

# # Display the updated dataframe with win streaks
# print(d_df_full2[['strHomeTeam', 'strAwayTeam', 'win streak home', 'win streak away']])

In [15]:

# Sort by date to ensure chronological order
d_df_full2['dateEvent'] = pd.to_datetime(d_df_full2['dateEvent'])
d_df_full2 = d_df_full2.sort_values('dateEvent')

# Initialize new columns for win streaks (last 5 games) for both home and away teams
d_df_full2["win streak home"] = 0
d_df_full2["win streak away"] = 0

# Create a dictionary to track win/loss history for each team
team_streaks = {}

# Iterate through the dataframe
for i, row in d_df_full2.iterrows():
    home_team = row['strHomeTeam']
    away_team = row['strAwayTeam']
    winner = row['winner']
    
    # Initialize tracking for home team if not already done
    if home_team not in team_streaks:
        team_streaks[home_team] = {'streaks': []}
    
    # Track the home team's result (win/loss/draw) in the streaks list
    if winner == home_team:
        team_streaks[home_team]['streaks'].append(1)  # Win
    elif winner is None:  # If draw
        team_streaks[home_team]['streaks'].append(0)  # Draw
    else:
        team_streaks[home_team]['streaks'].append(-1)  # Loss

    # Initialize tracking for away team if not already done
    if away_team not in team_streaks:
        team_streaks[away_team] = {'streaks': []}
    
    # Track the away team's result (win/loss/draw) in the streaks list
    if winner == away_team:
        team_streaks[away_team]['streaks'].append(1)  # Win
    elif winner is None:  # If draw
        team_streaks[away_team]['streaks'].append(0)  # Draw
    else:
        team_streaks[away_team]['streaks'].append(-1)  # Loss
    
    # Keep only the last 5 results for each team (home + away games)
    if len(team_streaks[home_team]['streaks']) > 5:
        team_streaks[home_team]['streaks'].pop(0)

    if len(team_streaks[away_team]['streaks']) > 5:
        team_streaks[away_team]['streaks'].pop(0)

    # Calculate the number of wins in the last 5 games for the home team
    d_df_full2.at[i, "win streak home last 5 games"] = sum(1 for x in team_streaks[home_team]['streaks'] if x == 1)

    # Calculate the number of wins in the last 5 games for the away team
    d_df_full2.at[i, "win streak away last 5 games"] = sum(1 for x in team_streaks[away_team]['streaks'] if x == 1)

# Display the updated dataframe with win streaks for home and away teams
print(d_df_full2[['strHomeTeam', 'strAwayTeam', 'win streak home last 5 games', 'win streak away last 5 games']])

           strHomeTeam        strAwayTeam  win streak home last 5 games  \
185          Leicester           Brighton                           0.0   
187        Southampton        Aston Villa                           0.0   
188           West Ham            Everton                           1.0   
190            Arsenal  Manchester United                           1.0   
191              Leeds          Brentford                           0.0   
..                 ...                ...                           ...   
962           West Ham     Crystal Palace                           2.0   
965  Manchester United           Brighton                           1.0   
963            Everton          Tottenham                           1.0   
964            Ipswich    Manchester City                           1.0   
966  Nottingham Forest        Southampton                           4.0   

     win streak away last 5 games  
185                           0.0  
187                        

In [16]:
d_df_full2[['win streak home last 5 games', 'win streak away last 5 games']] = d_df_full2[['win streak home last 5 games', 'win streak away last 5 games']].astype(int)

d_df_full2 = d_df_full2.drop(columns = ["win streak home","win streak away"])

d_df_full2

,strEvent,strHomeTeam,strAwayTeam,intHomeScore,intRound,intAwayScore,dateEvent,strTime,strVenue,Shots on Goal Home,...,Passes accurate Home,Passes accurate Away,Passes % Home,Passes % Away,expected_goals Home,expected_goals Away,winner,homeResult,win streak home last 5 games,win streak away last 5 games
185,Leicester vs Brighton,Leicester,Brighton,2,21,2,2023-01-21,15:00:00,King Power Stadium,2,...,287,562,80,88,1,2,draw,0.0,0,0
187,Southampton vs Aston Villa,Southampton,Aston Villa,0,21,1,2023-01-21,15:00:00,St. Mary's Stadium,5,...,223,447,70,80,0,1,Aston Villa,-1.0,0,1
188,West Ham vs Everton,West Ham,Everton,2,21,0,2023-01-21,15:00:00,London Stadium,4,...,271,623,79,87,2,1,West Ham,1.0,1,0
190,Arsenal vs Manchester United,Arsenal,Manchester United,3,21,2,2023-01-22,16:30:00,Emirates Stadium,5,...,425,300,82,78,3,0,Arsenal,1.0,1,0
191,Leeds vs Brentford,Leeds,Brentford,0,21,0,2023-01-22,14:00:00,Elland Road,6,...,366,220,74,65,1,0,draw,0.0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
962,West Ham vs Crystal Palace,West Ham,Crystal Palace,0,22,2,2025-01-18,15:00:00,London Stadium,0,...,441,378,82,81,0,1,Crystal Palace,-1.0,2,3
965,Manchester United vs Brighton,Manchester United,Brighton,1,22,3,2025-01-19,14:00:00,Old Trafford,1,...,370,353,82,82,1,2,Brighton,-1.0,1,2
963,Everton vs Tottenham,Everton,Tottenham,3,22,2,2025-01-19,14:00:00,Goodison Park,6,...,269,518,82,86,2,1,Everton,1.0,1,0
964,Ipswich vs Manchester City,Ipswich,Manchester City,0,22,6,2025-01-19,16:30:00,Portman Road,4,...,302,682,87,92,1,3,Manchester City,-1.0,1,3


In [17]:
d_df_full2["homeResult"] = d_df_full2["homeResult"].astype(int)

final_teams_data = d_df_full2
# .drop(columns = ["intAwayScore", "intHomeScore"])

# final_teams_data  = d_df_full2.drop(columns = [])

final_teams_data
#label enocoding for team names
#take off scores 

#label 
team_data = list(final_teams_data["strHomeTeam"])
empty_list = []
final_teams_data["dateEvent"] = pd.to_datetime(final_teams_data["dateEvent"])

sig_cols = [
    ["Shots on Goal Home", "Shots on Goal Away"], 
    ["Passes % Home", "Passes % Away"], 
    ["Shots off Goal Home", "Shots off Goal Away"], 
    ["Total Shots Home", "Total Shots Away"],
    ["Blocked Shots Home", "Blocked Shots Away"],
    ["Fouls Home", "Fouls Away"],
    ["Corner Kicks Home", "Corner Kicks Away"],
    ["Ball Possession Home", "Ball Possession Away"],
    ['Yellow Cards Home', 'Yellow Cards Away'],
    ['Red Cards Home', 'Red Cards Away'],
    ['Goalkeeper Saves Home','Goalkeeper Saves Away'],
    ['intHomeScore', 'intAwayScore'], 
    # ["expected_goals Home", "expected_goals Away"]
]

for i, row in final_teams_data.iterrows():
    home=row["strHomeTeam"]
    away=row["strAwayTeam"]
    date = row["dateEvent"]
    win_streak_home = row["win streak home last 5 games"]
    win_streak_away = row["win streak away last 5 games"]
    #shots on goal (mean)
    home_data = final_teams_data.query(f"strHomeTeam == '{home}' and dateEvent < '{date}'")
    home_away_data = final_teams_data.query(f"strAwayTeam == '{home}' and dateEvent < '{date}'")
    # home_concat = pd.concat([home_data, home_away_data], ignore_index=True, axis = 0)
    # list(home_data["Shots on Goal Home"]) + list(home_away_data["Shots on Goal Away"])
    # SOGH = pd.Series(list(home_data["Shots on Goal Home"]) + list(home_away_data["Shots on Goal Away"])).astype(int)

    away_data = final_teams_data.query(f"strAwayTeam == '{away}' and dateEvent < '{date}'")
    away_home_data_2 = final_teams_data.query(f"strHomeTeam == '{away}' and dateEvent < '{date}'")
    # home_concat_2 = pd.concat([away_data, away_home_data_2], ignore_index=True, axis = 0)
    # list(home_data["Shots on Goal Home"]) + list(home_away_data["Shots on Goal Away"])
    # SOGA = pd.Series(list(away_data["Shots on Goal Home"]) + list(away_home_data_2["Shots on Goal Away"])).astype(int)
    
    datum = {
        "Home": home, 
        "Away": away, 
        "Date": date, 
        "win streak away": win_streak_away,
        "win streak home":win_streak_home, 
        "winner": row["winner"],
        "Home result": row["homeResult"]
    }

    #pass accuracy (mean)
    for cols in sig_cols:
        home_metric = pd.Series(list(home_data[cols[0]]) + list(home_away_data[cols[1]])).astype(int)
        away_metric = pd.Series(list(away_data[cols[0]]) + list(away_home_data_2[cols[1]])).astype(int)
        datum[f'Hist {cols[0]} AVG'] = home_metric.mean()
        datum[f'Hist {cols[1]} AVG'] = away_metric.mean()




   
    empty_list.append(datum)

empty_df2 = pd.DataFrame(empty_list)
empty_df2

,Home,Away,Date,win streak away,win streak home,winner,Home result,Hist Shots on Goal Home AVG,Hist Shots on Goal Away AVG,Hist Passes % Home AVG,...,Hist Ball Possession Home AVG,Hist Ball Possession Away AVG,Hist Yellow Cards Home AVG,Hist Yellow Cards Away AVG,Hist Red Cards Home AVG,Hist Red Cards Away AVG,Hist Goalkeeper Saves Home AVG,Hist Goalkeeper Saves Away AVG,Hist intHomeScore AVG,Hist intAwayScore AVG
0,Leicester,Brighton,2023-01-21,0,0,draw,0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Southampton,Aston Villa,2023-01-21,1,0,Aston Villa,-1,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,West Ham,Everton,2023-01-21,0,1,West Ham,1,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Arsenal,Manchester United,2023-01-22,0,1,Arsenal,1,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Leeds,Brentford,2023-01-22,0,0,draw,0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
766,West Ham,Crystal Palace,2025-01-18,3,2,Crystal Palace,-1,4.038462,4.025974,79.269231,...,41.282051,55.896104,1.897436,2.051948,0.064103,0.064935,3.910256,2.948052,1.461538,1.402597
767,Manchester United,Brighton,2025-01-19,2,1,Brighton,-1,5.272727,4.052632,84.090909,...,51.727273,40.460526,2.051948,2.539474,0.064935,0.092105,3.181818,4.118421,1.415584,1.500000
768,Everton,Tottenham,2025-01-19,0,1,Everton,1,4.053333,4.513514,75.720000,...,40.160000,41.878378,2.066667,2.432432,0.053333,0.094595,3.093333,3.472973,0.946667,1.662162
769,Ipswich,Manchester City,2025-01-19,3,1,Manchester City,-1,3.238095,3.041096,78.952381,...,40.666667,34.534247,2.714286,2.287671,0.095238,0.041096,3.809524,4.246575,0.952381,1.000000


In [ ]:
empty_df2 = empty_df2.dropna()

empty_df2 = empty_df2.drop(columns = ["Hist Red Cards Home AVG", "Hist Red Cards Away AVG"])

empty_df2

,Home,Away,Date,win streak away,win streak home,winner,Home result,Hist Shots on Goal Home AVG,Hist Shots on Goal Away AVG,Hist Passes % Home AVG,...,Hist Corner Kicks Home AVG,Hist Corner Kicks Away AVG,Hist Ball Possession Home AVG,Hist Ball Possession Away AVG,Hist Yellow Cards Home AVG,Hist Yellow Cards Away AVG,Hist Goalkeeper Saves Home AVG,Hist Goalkeeper Saves Away AVG,Hist intHomeScore AVG,Hist intAwayScore AVG
5,Brentford,Southampton,2023-02-04,0,1,Brentford,1,0.000000,5.000000,65.000000,...,3.000000,2.000000,41.000000,63.000000,2.000000,2.000000,6.000000,5.000000,0.000000,1.000000
9,Manchester United,Leeds,2023-02-08,0,1,draw,0,4.500000,0.000000,82.000000,...,4.500000,3.000000,48.000000,41.000000,1.500000,2.000000,2.000000,6.000000,2.000000,0.000000
12,Southampton,Wolves,2023-02-11,2,0,Wolves,-1,5.500000,4.000000,73.000000,...,3.000000,7.000000,44.000000,58.000000,1.500000,1.000000,2.000000,4.000000,0.000000,0.000000
14,Crystal Palace,Brighton,2023-02-11,0,0,draw,0,3.000000,2.000000,80.000000,...,5.000000,5.000000,46.000000,36.000000,2.000000,0.000000,3.000000,2.000000,1.000000,2.000000
15,Arsenal,Brentford,2023-02-11,1,1,draw,0,5.000000,6.000000,82.000000,...,12.000000,3.000000,58.000000,55.000000,1.000000,2.000000,2.000000,0.000000,3.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
766,West Ham,Crystal Palace,2025-01-18,3,2,Crystal Palace,-1,4.038462,4.025974,79.269231,...,4.730769,5.103896,41.282051,55.896104,1.897436,2.051948,3.910256,2.948052,1.461538,1.402597
767,Manchester United,Brighton,2025-01-19,2,1,Brighton,-1,5.272727,4.052632,84.090909,...,5.714286,4.263158,51.727273,40.460526,2.051948,2.539474,3.181818,4.118421,1.415584,1.500000
768,Everton,Tottenham,2025-01-19,0,1,Everton,1,4.053333,4.513514,75.720000,...,4.573333,6.040541,40.160000,41.878378,2.066667,2.432432,3.093333,3.472973,0.946667,1.662162
769,Ipswich,Manchester City,2025-01-19,3,1,Manchester City,-1,3.238095,3.041096,78.952381,...,4.190476,3.068493,40.666667,34.534247,2.714286,2.287671,3.809524,4.246575,0.952381,1.000000


# PREPROCCESSING DATA

In [19]:
#PCA, Scaling, Synthetic data: SMOTE, More Columns, Removing NANS, regularization, coorelation map: What features have higher coorelation to Target (y). Regularization: Ridge & Lasso 

In [20]:
train = empty_df2[empty_df2["Date"] < '2024-12-5']

test = empty_df2[empty_df2["Date"] >= '2024-12-5']

In [21]:
# train_for_dummy = empty_df2[empty_df2["Date"] < '2024-12-5']

# test_for_dummy = empty_df2[empty_df2["Date"] >= '2024-12-5']

In [22]:
# test_for_dummy= test_for_dummy.drop(columns = [ "Date", "winner"])

# train_for_dummy = train_for_dummy.drop(columns = ["Date", "winner"])

In [23]:
train = train.drop(columns = ["Date", "winner", "Home", "Away"])

test = test.drop(columns = [ "Date", "winner", "Home", "Away"])



In [24]:
test

,win streak away,win streak home,Home result,Hist Shots on Goal Home AVG,Hist Shots on Goal Away AVG,Hist Passes % Home AVG,Hist Passes % Away AVG,Hist Shots off Goal Home AVG,Hist Shots off Goal Away AVG,Hist Total Shots Home AVG,...,Hist Corner Kicks Home AVG,Hist Corner Kicks Away AVG,Hist Ball Possession Home AVG,Hist Ball Possession Away AVG,Hist Yellow Cards Home AVG,Hist Yellow Cards Away AVG,Hist Goalkeeper Saves Home AVG,Hist Goalkeeper Saves Away AVG,Hist intHomeScore AVG,Hist intAwayScore AVG
691,2,3,1,4.552239,4.378788,78.149254,78.666667,4.835821,4.590909,13.373134,...,5.552239,6.030303,42.970149,41.424242,2.194030,2.424242,3.716418,3.575758,1.328358,1.590909
692,2,3,1,4.462687,4.132353,83.492537,80.647059,4.761194,4.117647,12.776119,...,5.462687,4.132353,50.611940,39.588235,2.089552,2.573529,3.373134,4.102941,1.432836,1.500000
693,2,2,-1,5.300000,4.588235,84.157143,84.279412,5.171429,5.191176,14.857143,...,5.728571,6.455882,51.342857,61.323529,2.085714,2.176471,3.257143,2.573529,1.428571,1.676471
694,0,2,1,4.528571,5.272727,85.914286,82.333333,4.728571,6.000000,12.728571,...,5.614286,6.151515,51.457143,51.636364,2.342857,1.878788,2.928571,2.545455,1.785714,2.090909
695,1,3,1,4.450704,4.842857,75.957746,79.571429,4.394366,4.657143,11.845070,...,4.549296,5.100000,45.169014,48.085714,1.971831,2.514286,3.704225,3.128571,1.535211,1.442857
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
766,3,2,-1,4.038462,4.025974,79.269231,83.974026,4.910256,4.623377,12.230769,...,4.730769,5.103896,41.282051,55.896104,1.897436,2.051948,3.910256,2.948052,1.461538,1.402597
767,2,1,-1,5.272727,4.052632,84.090909,80.644737,5.090909,4.052632,14.896104,...,5.714286,4.263158,51.727273,40.460526,2.051948,2.539474,3.181818,4.118421,1.415584,1.500000
768,0,1,1,4.053333,4.513514,75.720000,78.864865,5.360000,4.635135,12.653333,...,4.573333,6.040541,40.160000,41.878378,2.066667,2.432432,3.093333,3.472973,0.946667,1.662162
769,3,1,-1,3.238095,3.041096,78.952381,81.602740,3.380952,2.917808,9.380952,...,4.190476,3.068493,40.666667,34.534247,2.714286,2.287671,3.809524,4.246575,0.952381,1.000000


# MODELING EXPERIMENTS: RANDOMFOREST, NEURAL NETWORKS, XGBOOST

In [25]:
# dummy_train = pd.get_dummies(train_for_dummy).astype(int)

# dummy_test = pd.get_dummies(test_for_dummy).astype(int)

# dummy_test
# train_columns = dummy_train.columns

# # Apply the same column structure to the test set (ensure that the test set has all columns from the train set)
# dummy_test = dummy_test.reindex(columns=train_columns, fill_value=0)




In [26]:
# smote = SMOTE(random_state=42)

# dummy_X_train = dummy_train.drop(columns = "Home result")

# dummy_y_train = dummy_train["Home result"]

# dummy_X_test = dummy_test.drop(columns = "Home result")

# dummy_y_test = dummy_test["Home result"]



# dummy_X_train_resample, dummy_y_train_resample = smote.fit_resample(dummy_X_train, dummy_y_train)

# dummy_y_train_resample.value_counts()

In [27]:

X_train = train.drop(columns = "Home result")

y_train = train["Home result"]

X_test = test.drop(columns = "Home result")

y_test = test["Home result"]

X_test

,win streak away,win streak home,Hist Shots on Goal Home AVG,Hist Shots on Goal Away AVG,Hist Passes % Home AVG,Hist Passes % Away AVG,Hist Shots off Goal Home AVG,Hist Shots off Goal Away AVG,Hist Total Shots Home AVG,Hist Total Shots Away AVG,...,Hist Corner Kicks Home AVG,Hist Corner Kicks Away AVG,Hist Ball Possession Home AVG,Hist Ball Possession Away AVG,Hist Yellow Cards Home AVG,Hist Yellow Cards Away AVG,Hist Goalkeeper Saves Home AVG,Hist Goalkeeper Saves Away AVG,Hist intHomeScore AVG,Hist intAwayScore AVG
691,2,3,4.552239,4.378788,78.149254,78.666667,4.835821,4.590909,13.373134,12.515152,...,5.552239,6.030303,42.970149,41.424242,2.194030,2.424242,3.716418,3.575758,1.328358,1.590909
692,2,3,4.462687,4.132353,83.492537,80.647059,4.761194,4.117647,12.776119,11.720588,...,5.462687,4.132353,50.611940,39.588235,2.089552,2.573529,3.373134,4.102941,1.432836,1.500000
693,2,2,5.300000,4.588235,84.157143,84.279412,5.171429,5.191176,14.857143,14.014706,...,5.728571,6.455882,51.342857,61.323529,2.085714,2.176471,3.257143,2.573529,1.428571,1.676471
694,0,2,4.528571,5.272727,85.914286,82.333333,4.728571,6.000000,12.728571,16.727273,...,5.614286,6.151515,51.457143,51.636364,2.342857,1.878788,2.928571,2.545455,1.785714,2.090909
695,1,3,4.450704,4.842857,75.957746,79.571429,4.394366,4.657143,11.845070,13.042857,...,4.549296,5.100000,45.169014,48.085714,1.971831,2.514286,3.704225,3.128571,1.535211,1.442857
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
766,3,2,4.038462,4.025974,79.269231,83.974026,4.910256,4.623377,12.230769,11.844156,...,4.730769,5.103896,41.282051,55.896104,1.897436,2.051948,3.910256,2.948052,1.461538,1.402597
767,2,1,5.272727,4.052632,84.090909,80.644737,5.090909,4.052632,14.896104,11.565789,...,5.714286,4.263158,51.727273,40.460526,2.051948,2.539474,3.181818,4.118421,1.415584,1.500000
768,0,1,4.053333,4.513514,75.720000,78.864865,5.360000,4.635135,12.653333,12.783784,...,4.573333,6.040541,40.160000,41.878378,2.066667,2.432432,3.093333,3.472973,0.946667,1.662162
769,3,1,3.238095,3.041096,78.952381,81.602740,3.380952,2.917808,9.380952,8.219178,...,4.190476,3.068493,40.666667,34.534247,2.714286,2.287671,3.809524,4.246575,0.952381,1.000000


In [28]:
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

# # scaler =  StandardScaler()

# # X_scaler = scaler.fit(X_train)

# # X_train_scaled = X_scaler.transform(X_train)
# # X_test_scaled = X_scaler.transform(X_test)

y_train_resampled.value_counts()

Home result
 1    315
 0    315
-1    315
Name: count, dtype: int64

In [29]:
# #Random Forest Classification

# rf_model_d = RandomForestClassifier(n_estimators=100, random_state=1, class_weight="balanced")
# rf_model_d = rf_model_d.fit(dummy_X_train_resample, dummy_y_train_resample)
# predictions_d = rf_model_d.predict(dummy_X_test)

# y2_d = dummy_y_test.values.flatten()

# p2_d = predictions_d.flatten()

# cm_d = confusion_matrix(y2_d, p2_d)
# cm_df_d = pd.DataFrame(
#     cm_d, index=["Actual 1", "Actual 0", "Actual -1"], columns=["Predicted 1", "Predicted 0", "Predictied -1"]
# )

# # # Calculating the accuracy score
# acc_score = accuracy_score(dummy_y_test, predictions_d)

# # print("Confusion Matrix")
# # display(cm_df)
# print(f"Accuracy Score : {acc_score}")
# # print("Classification Report")
# print(classification_report(dummy_y_test, predictions_d))

# importances = rf_model_d.feature_importances_
# # List the top 10 most important features
# # importances_sorted = sorted(zip(rf_model.feature_importances_, .columns), reverse=True)
# # importances_sorted[:10]

# cm_df_d

In [30]:
#Random Forest Classification

rf_model = RandomForestClassifier(n_estimators=100, random_state=1, class_weight="balanced")
rf_model = rf_model.fit(X_train_resampled, y_train_resampled)
predictions = rf_model.predict(X_test)

y2 = y_test.values.flatten()

p2 = predictions.flatten()

cm = confusion_matrix(y2, p2)
cm_df = pd.DataFrame(
    cm, index=["Actual -1", "Actual 0", "Actual 1"], columns=["Predicted -1", "Predicted 0", "Predictied 1"]
)

# # Calculating the accuracy score
acc_score = accuracy_score(y_test, predictions)

# print("Confusion Matrix")
# display(cm_df)
print(f"Accuracy Score : {acc_score}")
# print("Classification Report")
print(classification_report(y_test, predictions))

importances = rf_model.feature_importances_
# List the top 10 most important features
importances_sorted = sorted(zip(rf_model.feature_importances_, X_train_resampled.columns), reverse=True)
print(importances_sorted[:10])

cm_df

Accuracy Score : 0.5625
              precision    recall  f1-score   support

          -1       0.62      0.66      0.64        32
           0       0.33      0.22      0.26        23
           1       0.61      0.76      0.68        25

    accuracy                           0.56        80
   macro avg       0.52      0.54      0.53        80
weighted avg       0.53      0.56      0.54        80

[(0.10136069680284106, 'win streak home'), (0.07782876462385575, 'win streak away'), (0.04713208942364209, 'Hist Ball Possession Home AVG'), (0.043796243745039945, 'Hist intHomeScore AVG'), (0.042274506528054735, 'Hist Passes % Home AVG'), (0.04108027784305904, 'Hist Shots off Goal Away AVG'), (0.03878435035121703, 'Hist Total Shots Away AVG'), (0.0386202583960652, 'Hist Fouls Home AVG'), (0.03831496023023649, 'Hist Shots off Goal Home AVG'), (0.038288882869312646, 'Hist Corner Kicks Away AVG')]


,Predicted -1,Predicted 0,Predictied 1
Actual -1,21,7,4
Actual 0,10,5,8
Actual 1,3,3,19


In [31]:
import pickle 

with open("prem_pred.pk1", "wb")as proj4_file:
    pickle.dump(rf_model, proj4_file)

In [32]:
d_df2_new_data = d_df.loc[d_df["strStatus"] == "Not Started"].reset_index(drop = True)

d_df2_new_data

,idEvent,idAPIfootball,strEvent,strFilename,strSport,idLeague,strSeason,strHomeTeam,strAwayTeam,intHomeScore,...,dateEvent,strTime,idHomeTeam,idAwayTeam,intScore,idVenue,strVenue,strCountry,strVideo,strStatus
0,2070058,1208243,Bournemouth vs Nottingham Forest,English Premier League 2025-01-25 Bournemouth ...,Soccer,4328,2024-2025,Bournemouth,Nottingham Forest,None,...,2025-01-25,15:00:00,134301,133720,None,16113,Vitality Stadium,England,,Not Started
1,2070060,1208245,Brighton vs Everton,English Premier League 2025-01-25 Brighton vs ...,Soccer,4328,2024-2025,Brighton,Everton,None,...,2025-01-25,15:00:00,133619,133615,None,0,,England,,Not Started
2,2070063,1208248,Liverpool vs Ipswich,English Premier League 2025-01-25 Liverpool vs...,Soccer,4328,2024-2025,Liverpool,Ipswich,None,...,2025-01-25,15:00:00,133602,133622,None,15407,Anfield,England,,Not Started
3,2070064,1208249,Manchester City vs Chelsea,English Premier League 2025-01-25 Manchester C...,Soccer,4328,2024-2025,Manchester City,Chelsea,None,...,2025-01-25,17:30:00,133613,133610,None,16119,Etihad Stadium,England,,Not Started
4,2070065,1208250,Southampton vs Newcastle,English Premier League 2025-01-25 Southampton ...,Soccer,4328,2024-2025,Southampton,Newcastle,None,...,2025-01-25,15:00:00,134778,134777,None,22975,St. Mary,England,,Not Started
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
156,2070213,1208398,Newcastle vs Everton,English Premier League 2025-05-25 Newcastle vs...,Soccer,4328,2024-2025,Newcastle,Everton,None,...,2025-05-25,15:00:00,134777,133615,None,16294,St James Park,England,None,Not Started
157,2070214,1208399,Nottingham Forest vs Chelsea,English Premier League 2025-05-25 Nottingham F...,Soccer,4328,2024-2025,Nottingham Forest,Chelsea,None,...,2025-05-25,15:00:00,133720,133610,None,18249,City Ground,England,None,Not Started
158,2070215,1208400,Southampton vs Arsenal,English Premier League 2025-05-25 Southampton ...,Soccer,4328,2024-2025,Southampton,Arsenal,None,...,2025-05-25,15:00:00,134778,133604,None,16123,St. Mary,England,None,Not Started
159,2070216,1208401,Tottenham vs Brighton,English Premier League 2025-05-25 Tottenham vs...,Soccer,4328,2024-2025,Tottenham,Brighton,None,...,2025-05-25,15:00:00,133616,133619,None,16118,Tottenham Hotspur Stadium,England,None,Not Started


In [33]:
future_data_1 = pd.DataFrame(columns = d_df_full2.columns)

future_data_1["strHomeTeam"] = d_df2_new_data["strHomeTeam"]

future_data_1["strAwayTeam"] =d_df2_new_data["strAwayTeam"]

future_data_1["dateEvent"] = pd.to_datetime(d_df2_new_data["dateEvent"])

# for i, row in future_data_1.iterrows: 
#     home_team = row["Home"] 
#     away_team = row["Away"]

#     home_stats = latest_matchweek.loc[latest_matchweek["Home"] == home_team]
#     away_stats = latest_matchweek.loc[latest_matchweek["Away"] == away_team]

future_data_1 = future_data_1.iloc[:10]

future_data_1

,strEvent,strHomeTeam,strAwayTeam,intHomeScore,intRound,intAwayScore,dateEvent,strTime,strVenue,Shots on Goal Home,...,Passes accurate Home,Passes accurate Away,Passes % Home,Passes % Away,expected_goals Home,expected_goals Away,winner,homeResult,win streak home last 5 games,win streak away last 5 games
0,NaN,Bournemouth,Nottingham Forest,NaN,NaN,NaN,2025-01-25,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,Brighton,Everton,NaN,NaN,NaN,2025-01-25,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,Liverpool,Ipswich,NaN,NaN,NaN,2025-01-25,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,Manchester City,Chelsea,NaN,NaN,NaN,2025-01-25,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,Southampton,Newcastle,NaN,NaN,NaN,2025-01-25,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,Wolves,Arsenal,NaN,NaN,NaN,2025-01-25,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,NaN,Aston Villa,West Ham,NaN,NaN,NaN,2025-01-26,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,NaN,Crystal Palace,Brentford,NaN,NaN,NaN,2025-01-26,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,NaN,Fulham,Manchester United,NaN,NaN,NaN,2025-01-26,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,NaN,Tottenham,Leicester,NaN,NaN,NaN,2025-01-26,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [34]:
d_df_full_3 = pd.concat([d_df_full2,future_data_1]).reset_index(drop = True)

d_df_full_3

,strEvent,strHomeTeam,strAwayTeam,intHomeScore,intRound,intAwayScore,dateEvent,strTime,strVenue,Shots on Goal Home,...,Passes accurate Home,Passes accurate Away,Passes % Home,Passes % Away,expected_goals Home,expected_goals Away,winner,homeResult,win streak home last 5 games,win streak away last 5 games
0,Leicester vs Brighton,Leicester,Brighton,2,21,2,2023-01-21,15:00:00,King Power Stadium,2,...,287,562,80,88,1,2,draw,0,0,0
1,Southampton vs Aston Villa,Southampton,Aston Villa,0,21,1,2023-01-21,15:00:00,St. Mary's Stadium,5,...,223,447,70,80,0,1,Aston Villa,-1,0,1
2,West Ham vs Everton,West Ham,Everton,2,21,0,2023-01-21,15:00:00,London Stadium,4,...,271,623,79,87,2,1,West Ham,1,1,0
3,Arsenal vs Manchester United,Arsenal,Manchester United,3,21,2,2023-01-22,16:30:00,Emirates Stadium,5,...,425,300,82,78,3,0,Arsenal,1,1,0
4,Leeds vs Brentford,Leeds,Brentford,0,21,0,2023-01-22,14:00:00,Elland Road,6,...,366,220,74,65,1,0,draw,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
776,NaN,Wolves,Arsenal,NaN,NaN,NaN,2025-01-25,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
777,NaN,Aston Villa,West Ham,NaN,NaN,NaN,2025-01-26,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
778,NaN,Crystal Palace,Brentford,NaN,NaN,NaN,2025-01-26,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
779,NaN,Fulham,Manchester United,NaN,NaN,NaN,2025-01-26,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [35]:

# Sort by date to ensure chronological order
d_df_full_3['dateEvent'] = pd.to_datetime(d_df_full_3['dateEvent'])
d_df_full_3 = d_df_full_3.sort_values('dateEvent')

# Initialize new columns for win streaks (last 5 games) for both home and away teams
d_df_full_3["win streak home"] = 0
d_df_full_3["win streak away"] = 0

# Create a dictionary to track win/loss history for each team
team_streaks = {}

# Iterate through the dataframe
for i, row in d_df_full_3.iterrows():
    home_team = row['strHomeTeam']
    away_team = row['strAwayTeam']
    winner = row['winner']
    
    # Initialize tracking for home team if not already done
    if home_team not in team_streaks:
        team_streaks[home_team] = {'streaks': []}
    
    # Track the home team's result (win/loss/draw) in the streaks list
    if winner == home_team:
        team_streaks[home_team]['streaks'].append(1)  # Win
    elif winner is None:  # If draw
        team_streaks[home_team]['streaks'].append(0)  # Draw
    else:
        team_streaks[home_team]['streaks'].append(-1)  # Loss

    # Initialize tracking for away team if not already done
    if away_team not in team_streaks:
        team_streaks[away_team] = {'streaks': []}
    
    # Track the away team's result (win/loss/draw) in the streaks list
    if winner == away_team:
        team_streaks[away_team]['streaks'].append(1)  # Win
    elif winner is None:  # If draw
        team_streaks[away_team]['streaks'].append(0)  # Draw
    else:
        team_streaks[away_team]['streaks'].append(-1)  # Loss
    
    # Keep only the last 5 results for each team (home + away games)
    if len(team_streaks[home_team]['streaks']) > 5:
        team_streaks[home_team]['streaks'].pop(0)

    if len(team_streaks[away_team]['streaks']) > 5:
        team_streaks[away_team]['streaks'].pop(0)

    # Calculate the number of wins in the last 5 games for the home team
    d_df_full_3.at[i, "win streak home last 5 games"] = sum(1 for x in team_streaks[home_team]['streaks'] if x == 1)

    # Calculate the number of wins in the last 5 games for the away team
    d_df_full_3.at[i, "win streak away last 5 games"] = sum(1 for x in team_streaks[away_team]['streaks'] if x == 1)

# Display the updated dataframe with win streaks for home and away teams
print(d_df_full_3[['strHomeTeam', 'strAwayTeam', 'win streak home last 5 games', 'win streak away last 5 games']])

d_df_full_3

        strHomeTeam        strAwayTeam win streak home last 5 games  \
0         Leicester           Brighton                            0   
1       Southampton        Aston Villa                            0   
2          West Ham            Everton                            1   
3           Arsenal  Manchester United                            1   
4             Leeds          Brentford                            0   
..              ...                ...                          ...   
773       Liverpool            Ipswich                            2   
778  Crystal Palace          Brentford                            3   
779          Fulham  Manchester United                            1   
777     Aston Villa           West Ham                            2   
780       Tottenham          Leicester                            0   

    win streak away last 5 games  
0                              0  
1                              1  
2                              0  
3      

,strEvent,strHomeTeam,strAwayTeam,intHomeScore,intRound,intAwayScore,dateEvent,strTime,strVenue,Shots on Goal Home,...,Passes % Home,Passes % Away,expected_goals Home,expected_goals Away,winner,homeResult,win streak home last 5 games,win streak away last 5 games,win streak home,win streak away
0,Leicester vs Brighton,Leicester,Brighton,2,21,2,2023-01-21,15:00:00,King Power Stadium,2,...,80,88,1,2,draw,0,0,0,0,0
1,Southampton vs Aston Villa,Southampton,Aston Villa,0,21,1,2023-01-21,15:00:00,St. Mary's Stadium,5,...,70,80,0,1,Aston Villa,-1,0,1,0,0
2,West Ham vs Everton,West Ham,Everton,2,21,0,2023-01-21,15:00:00,London Stadium,4,...,79,87,2,1,West Ham,1,1,0,0,0
3,Arsenal vs Manchester United,Arsenal,Manchester United,3,21,2,2023-01-22,16:30:00,Emirates Stadium,5,...,82,78,3,0,Arsenal,1,1,0,0,0
4,Leeds vs Brentford,Leeds,Brentford,0,21,0,2023-01-22,14:00:00,Elland Road,6,...,74,65,1,0,draw,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
773,NaN,Liverpool,Ipswich,NaN,NaN,NaN,2025-01-25,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,2,1,0,0
778,NaN,Crystal Palace,Brentford,NaN,NaN,NaN,2025-01-26,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,3,1,0,0
779,NaN,Fulham,Manchester United,NaN,NaN,NaN,2025-01-26,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,1,1,0,0
777,NaN,Aston Villa,West Ham,NaN,NaN,NaN,2025-01-26,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,2,1,0,0


In [36]:


final_teams_data_2 = d_df_full_3
# .drop(columns = ["intAwayScore", "intHomeScore"])

# final_teams_data  = d_df_full2.drop(columns = [])

final_teams_data_2
#label enocoding for team names
#take off scores 

#label 
team_data = list(final_teams_data_2["strHomeTeam"])
empty_list = []
final_teams_data_2["dateEvent"] = pd.to_datetime(final_teams_data_2["dateEvent"])

sig_cols = [
    ["Shots on Goal Home", "Shots on Goal Away"], 
    ["Passes % Home", "Passes % Away"], 
    ["Shots off Goal Home", "Shots off Goal Away"], 
    ["Total Shots Home", "Total Shots Away"],
    ["Blocked Shots Home", "Blocked Shots Away"],
    ["Fouls Home", "Fouls Away"],
    ["Corner Kicks Home", "Corner Kicks Away"],
    ["Ball Possession Home", "Ball Possession Away"],
    ['Yellow Cards Home', 'Yellow Cards Away'],
    ['Goalkeeper Saves Home','Goalkeeper Saves Away'],
    ['intHomeScore', 'intAwayScore'], 
    # ["expected_goals Home", "expected_goals Away"]
]

for i, row in final_teams_data_2.iterrows():
    home=row["strHomeTeam"]
    away=row["strAwayTeam"]
    date = row["dateEvent"]
    win_streak_home = row["win streak home last 5 games"]
    win_streak_away = row["win streak away last 5 games"]
    #shots on goal (mean)
    home_data = final_teams_data_2.query(f"strHomeTeam == '{home}' and dateEvent < '{date}'")
    home_away_data = final_teams_data_2.query(f"strAwayTeam == '{home}' and dateEvent < '{date}'")
    # home_concat = pd.concat([home_data, home_away_data], ignore_index=True, axis = 0)
    # list(home_data["Shots on Goal Home"]) + list(home_away_data["Shots on Goal Away"])
    # SOGH = pd.Series(list(home_data["Shots on Goal Home"]) + list(home_away_data["Shots on Goal Away"])).astype(int)

    away_data = final_teams_data_2.query(f"strAwayTeam == '{away}' and dateEvent < '{date}'")
    away_home_data_2 = final_teams_data_2.query(f"strHomeTeam == '{away}' and dateEvent < '{date}'")
    # home_concat_2 = pd.concat([away_data, away_home_data_2], ignore_index=True, axis = 0)
    # list(home_data["Shots on Goal Home"]) + list(home_away_data["Shots on Goal Away"])
    # SOGA = pd.Series(list(away_data["Shots on Goal Home"]) + list(away_home_data_2["Shots on Goal Away"])).astype(int)
    
    datum = {
        "Home": home, 
        "Away": away, 
        "Date": date, 
        "win streak away": win_streak_away,
        "win streak home":win_streak_home, 
        "winner": row["winner"],
        "Home result": row["homeResult"]
    }

    #pass accuracy (mean)
    for cols in sig_cols:
        home_metric = pd.Series(list(home_data[cols[0]]) + list(home_away_data[cols[1]])).astype(int)
        away_metric = pd.Series(list(away_data[cols[0]]) + list(away_home_data_2[cols[1]])).astype(int)
        datum[f'Hist {cols[0]} AVG'] = home_metric.mean()
        datum[f'Hist {cols[1]} AVG'] = away_metric.mean()




   
    empty_list.append(datum)

empty_df_3 = pd.DataFrame(empty_list)
empty_df_3

,Home,Away,Date,win streak away,win streak home,winner,Home result,Hist Shots on Goal Home AVG,Hist Shots on Goal Away AVG,Hist Passes % Home AVG,...,Hist Corner Kicks Home AVG,Hist Corner Kicks Away AVG,Hist Ball Possession Home AVG,Hist Ball Possession Away AVG,Hist Yellow Cards Home AVG,Hist Yellow Cards Away AVG,Hist Goalkeeper Saves Home AVG,Hist Goalkeeper Saves Away AVG,Hist intHomeScore AVG,Hist intAwayScore AVG
0,Leicester,Brighton,2023-01-21,0,0,draw,0.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Southampton,Aston Villa,2023-01-21,1,0,Aston Villa,-1.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,West Ham,Everton,2023-01-21,0,1,West Ham,1.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Arsenal,Manchester United,2023-01-22,0,1,Arsenal,1.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Leeds,Brentford,2023-01-22,0,0,draw,0.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
776,Liverpool,Ipswich,2025-01-25,1,2,NaN,NaN,6.597403,5.727273,85.428571,...,6.844156,6.681818,60.987013,59.727273,1.948052,1.590909,2.714286,2.272727,2.285714,1.954545
777,Crystal Palace,Brentford,2025-01-26,1,3,NaN,NaN,4.333333,5.367089,77.500000,...,4.910256,5.987342,43.538462,54.101266,1.910256,1.911392,2.717949,2.962025,1.333333,1.544304
778,Fulham,Manchester United,2025-01-26,1,1,NaN,NaN,4.407895,4.589744,83.421053,...,5.223684,6.064103,50.789474,47.692308,2.105263,1.974359,3.328947,3.782051,1.473684,1.423077
779,Aston Villa,West Ham,2025-01-26,1,2,NaN,NaN,4.448718,5.810127,85.705128,...,5.692308,5.848101,51.102564,58.215190,2.256410,1.569620,2.910256,2.531646,1.743590,1.860759


In [37]:
empty_df_to_predict_on = empty_df_3.iloc[-10:]

# empty_df_team_columns = empty_df_to_predict_on[["Home", "Away", "Date"]]

# empty_df_to_predict_on =  empty_df_to_predict_on.drop(columns =["winner", "Home", "Away", "Home result", "Date"])

empty_df_to_predict_on

empty_df_to_predict_on.columns

Index(['Home', 'Away', 'Date', 'win streak away', 'win streak home', 'winner',
       'Home result', 'Hist Shots on Goal Home AVG',
       'Hist Shots on Goal Away AVG', 'Hist Passes % Home AVG',
       'Hist Passes % Away AVG', 'Hist Shots off Goal Home AVG',
       'Hist Shots off Goal Away AVG', 'Hist Total Shots Home AVG',
       'Hist Total Shots Away AVG', 'Hist Blocked Shots Home AVG',
       'Hist Blocked Shots Away AVG', 'Hist Fouls Home AVG',
       'Hist Fouls Away AVG', 'Hist Corner Kicks Home AVG',
       'Hist Corner Kicks Away AVG', 'Hist Ball Possession Home AVG',
       'Hist Ball Possession Away AVG', 'Hist Yellow Cards Home AVG',
       'Hist Yellow Cards Away AVG', 'Hist Goalkeeper Saves Home AVG',
       'Hist Goalkeeper Saves Away AVG', 'Hist intHomeScore AVG',
       'Hist intAwayScore AVG'],
      dtype='object')

In [38]:
home_columns = ['Home', 'Date', 'win streak home',
                      'Hist Shots on Goal Home AVG',
        'Hist Passes % Home AVG',
        'Hist Shots off Goal Home AVG',
        'Hist Total Shots Home AVG',
        'Hist Blocked Shots Home AVG',
        'Hist Fouls Home AVG',
        'Hist Corner Kicks Home AVG',
        'Hist Ball Possession Home AVG',
        'Hist Yellow Cards Home AVG',
       'Hist Goalkeeper Saves Home AVG',
      'Hist intHomeScore AVG',
    ]

renamer = {"Home": "team", "Away": "team",
           "win streak home": "win streak", "win streak away": "win streak",
           "Hist intHomeScore AVG": "Hist intScore AVG", "Hist intAwayScore AVG": "Hist intScore AVG"}

In [39]:
away_columns = [ 'Away', 'Date', 'win streak away',
       'Hist Shots on Goal Away AVG', 
       'Hist Passes % Away AVG', 
       'Hist Shots off Goal Away AVG', 
       'Hist Total Shots Away AVG', 
       'Hist Blocked Shots Away AVG', 
       'Hist Fouls Away AVG', 
       'Hist Corner Kicks Away AVG', 
       'Hist Ball Possession Away AVG', 
       'Hist Yellow Cards Away AVG', 
       'Hist Goalkeeper Saves Away AVG',
       'Hist intAwayScore AVG']


In [53]:
home_team_df_2 = empty_df_to_predict_on[home_columns].to_csv("../static/home_teams_data.csv")
away_team_df_2 = empty_df_to_predict_on[away_columns].to_csv("../static/away_teams_data.csv")
empty_df_to_predict_on.to_csv("../static/full_teams_original")

In [41]:
renamer = {"Home": "team", "Away": "team",
           "win streak home": "win streak", "win streak away": "win streak",
           "Hist intHomeScore AVG": "Hist intScore AVG", "Hist intAwayScore AVG": "Hist intScore AVG"}

def rmHomeAway(old):
    new = old.replace(" Away", "").replace(" Home", "")
    return new

####################################################################################################################################

def rmHomeAwayReversalHome(old2):
    new2 = old2.replace(" AVG", " Home AVG")
    return new2

def rmHomeAwayReversalAway(old2):
    new2 = old2.replace(" AVG", " Away AVG")
    return new2

renamer_reversal_home = {"team": "Home",
           "win streak": "win streak home",
           "Hist intScore Home AVG": "Hist intHomeScore AVG"}
renamer_reversal_away =  {"team": "Away",
         "win streak": "win streak away",
         "Hist intScore Away AVG": "Hist intAwayScore AVG"}

######################################################################################################################################



In [42]:
home_team_df = empty_df_to_predict_on[home_columns].rename(columns=renamer).rename(columns=rmHomeAway)
away_team_df = empty_df_to_predict_on[away_columns].rename(columns=renamer).rename(columns=rmHomeAway)

all(home_team_df.columns == away_team_df.columns)

True

In [1]:
full_team_df = pd.concat([home_team_df, away_team_df]).reset_index(drop=True)
print(full_team_df.shape)
full_team_df.head(3)

NameError: name 'pd' is not defined

In [44]:
full_team_df.to_csv("../static/teams_data.csv")

## OLD above
## NEW below (from CSV WEB APP PRACTICE)

In [45]:
teams_df = pd.read_csv("../static/teams_data.csv", index_col=0)
teams_df.head(2)

,team,Date,win streak,Hist Shots on Goal AVG,Hist Passes % AVG,Hist Shots off Goal AVG,Hist Total Shots AVG,Hist Blocked Shots AVG,Hist Fouls AVG,Hist Corner Kicks AVG,Hist Ball Possession AVG,Hist Yellow Cards AVG,Hist Goalkeeper Saves AVG,Hist intScore AVG
0,Southampton,2025-01-25,0,3.439024,82.195122,3.536585,9.951220,2.975610,12.073171,4.243902,48.463415,2.609756,3.243902,0.829268
1,Manchester City,2025-01-25,3,6.648649,90.135135,5.756757,17.486486,5.081081,7.729730,7.432432,65.135135,1.445946,2.040541,2.364865


In [46]:
empty_df_to_predict_on["Predictions_home"] = rf_model.predict(empty_df_to_predict_on)

empty_df_to_predict_on

ValueError: The feature names should match those that were passed during fit.
Feature names unseen at fit time:
- Away
- Date
- Home
- Home result
- winner


In [49]:
latest_matchweek = latest_matchweek.reset_index(drop=True)

latest_matchweek


,Home,Away,Date,win streak away,win streak home,winner,Home result,Hist Shots on Goal Home AVG,Hist Shots on Goal Away AVG,Hist Passes % Home AVG,...,Hist Corner Kicks Home AVG,Hist Corner Kicks Away AVG,Hist Ball Possession Home AVG,Hist Ball Possession Away AVG,Hist Yellow Cards Home AVG,Hist Yellow Cards Away AVG,Hist Goalkeeper Saves Home AVG,Hist Goalkeeper Saves Away AVG,Hist intHomeScore AVG,Hist intAwayScore AVG
0,Fulham,Bournemouth,2024-12-29,2,1,draw,0,4.416667,5.361111,83.361111,...,5.305556,6.083333,50.236111,56.222222,2.138889,2.138889,3.347222,3.263889,1.444444,1.611111
1,Leicester,Manchester City,2024-12-29,1,0,Manchester City,-1,3.500000,2.971429,78.055556,...,3.333333,3.071429,44.694444,33.971429,2.250000,2.342857,3.583333,4.242857,1.194444,1.000000
2,Crystal Palace,Southampton,2024-12-29,0,2,Crystal Palace,1,4.202703,5.270270,77.472973,...,4.891892,6.189189,43.540541,51.648649,1.986486,1.864865,2.783784,2.513514,1.310811,2.054054
3,Everton,Nottingham Forest,2024-12-29,5,1,Nottingham Forest,-1,4.152778,4.555556,75.708333,...,4.541667,6.430556,39.694444,61.472222,2.041667,2.166667,3.027778,2.569444,0.986111,1.625000
4,Tottenham,Wolves,2024-12-29,2,1,draw,0,5.521127,5.432432,85.619718,...,6.225352,6.216216,58.436620,51.067568,2.211268,2.310811,2.845070,2.716216,2.000000,1.797297
5,West Ham,Liverpool,2024-12-29,3,2,Liverpool,-1,4.106667,3.726027,79.053333,...,4.880000,3.917808,41.160000,38.520548,1.920000,2.369863,3.880000,4.205479,1.466667,1.082192
6,Aston Villa,Brighton,2024-12-30,0,2,draw,0,4.472973,4.082192,85.864865,...,5.675676,4.191781,51.094595,39.917808,2.297297,2.547945,2.945946,4.164384,1.743243,1.520548
7,Ipswich,Chelsea,2024-12-30,2,2,Ipswich,1,3.111111,4.763889,79.777778,...,4.500000,4.555556,42.055556,40.597222,2.611111,2.513889,3.944444,3.555556,0.888889,1.513889
8,Manchester United,Newcastle,2024-12-30,4,1,Newcastle,-1,5.297297,4.743243,84.162162,...,5.743243,5.000000,51.662162,47.689189,2.054054,2.500000,3.162162,3.175676,1.405405,1.418919
9,Brentford,Arsenal,2025-01-01,3,1,Arsenal,-1,4.453333,2.945946,76.120000,...,4.573333,3.054054,45.280000,40.972973,1.986667,2.229730,3.773333,3.351351,1.520000,0.945946


In [ ]:
import os
from dotenv import load_dotenv

load_dotenv() 
database_url = os.getenv("DATABASE_URL")
print(database_url)


engine = create_engine(database_url)

base = declarative_base()

class user_info(base): 
    __tablename__ = 'user_info'
    id = Column(Integer, primary_key=True, autoincrement=True, nullable=False)
    first_name = Column(String(250))
    last_name = Column(String(250))
    birthday = Column(Date)

    __table_args__ = (
        PrimaryKeyConstraint('id'), 
        {'extend_existing': True},
    )    
    
# base.metadata.create_all(engine)

In [348]:
from sqlalchemy.orm import sessionmaker
Session = sessionmaker(bind = engine)

session = Session()

# new = user_info(name = "Joe")

# session.add(new)
# session.commit()

In [349]:
class order(base): 
    __tablename__ = 'amount_add'
    
    order_id = Column(Integer, primary_key=True, autoincrement=True)
    user_id = Column(Integer, ForeignKey('user_info.id'))
    amount = Column(Float)
    team = Column(String(250))
    W_D_L = Column(String(250))

    __table_args__ = (
        ForeignKeyConstraint(['user_id'], ['user_info.id']),
        {'extend_existing':True},
    )
    
    
# base.metadata.create_all(engine)

In [350]:
# with engine.connect() as connection:
#     connection.execute(text("ALTER TABLE amount_add RENAME COLUMN reason_for TO team;"))
#     connection.commit()
#     print("Column name changed successfully!")

In [351]:
# try:
#     with engine.connect() as connection: 
#         connection.execute(text("ALTER TABLE amount_add ADD COLUMN Win_Draw_Loss VARCHAR(250);"))
#         print("success")
#         connection.commit()
# except Exception as e:
#     print(f"Error: {e}")

In [352]:
# new_order = order(user_id = 1 , amount= 1000)

# session.add(new_order)
# session.commit()

In [353]:
session.close()